In [2]:
import os
import csv
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

In [3]:
csv_path = "train.csv"
rows = []
with open(csv_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(row)

print(f"Wczytano {len(rows)} przykładów")

Wczytano 12672 przykładów


In [5]:
import cv2
import numpy as np

def lbp_hist(img_path, size=(64, 64)):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        raise ValueError("Nie można wczytać obrazu")

    img = cv2.resize(img, size)

    h, w = img.shape
    lbp = np.zeros((h, w), dtype=np.uint8)

    neighbors = [
        (-1, -1), (-1, 0), (-1, 1),
        (0, 1),
        (1, 1), (1, 0), (1, -1),
        (0, -1)
    ]

    for y in range(1, h - 1):
        for x in range(1, w - 1):
            center = img[y, x]
            code = 0

            for i, (dy, dx) in enumerate(neighbors):
                if img[y + dy, x + dx] >= center:
                    code |= (1 << i)

            lbp[y, x] = code

    hist = np.zeros(256, dtype=np.float32)

    for value in lbp.ravel():
        hist[value] += 1

    hist /= (hist.sum() + 1e-7)

    return hist

In [ ]:
import os
import cv2
import numpy as np

X_LBP = []
X_16 = []
X_32 = []
y = []
valid_paths = []
skipped = 0

for row in rows:
    img_path = row["image_path"]
    label = int(row["label"])

    if not os.path.exists(img_path):
        print(f"Pominięty nieistniejący: {img_path}")
        skipped += 1
        continue

    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"Pominięty uszkodzony: {img_path}")
        skipped += 1
        continue

    # LBP feature
    hist = lbp_hist(img_path)
    X_LBP.append(hist)

    # obrazy zeskalowane
    img_16 = cv2.resize(img, (16, 16))
    img_32 = cv2.resize(img, (32, 32))

    X_16.append(img_16.flatten())
    X_32.append(img_32.flatten())

    y.append(label)
    valid_paths.append(img_path)

# konwersja do numpy
X_LBP = np.array(X_LBP, dtype=np.float32)
X_16 = np.array(X_16, dtype=np.float32) / 255.0
X_32 = np.array(X_32, dtype=np.float32) / 255.0
y = np.array(y, dtype=np.int64)

print(f"W przygotowaniu: {len(y)} przykładów")
print(f"Pominięto: {skipped}")
print(f"X_LBP: {X_LBP.shape}")
print(f"X_16: {X_16.shape}")
print(f"X_32: {X_32.shape}")

W przygotowaniu: 12672 przykładów
Pominięto: 0
X_LBP: (12672, 256)
X_16: (12672, 256)
X_32: (12672, 1024)


In [13]:
len(np.unique(y))

13

In [7]:
from sklearn.model_selection import StratifiedShuffleSplit

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(sss.split(X_LBP, y))

X_train_LBP = X_LBP[train_idx]
X_test_LBP  = X_LBP[test_idx]
y_train = y[train_idx]
y_test  = y[test_idx]

X_train_16 = X_16[train_idx]
X_test_16  = X_16[test_idx]

X_train_32 = X_32[train_idx]
X_test_32  = X_32[test_idx]

In [8]:
print(X_train_LBP.shape, y_train.shape)
print(X_test_LBP.shape, y_test.shape)

(10137, 256) (10137,)
(2535, 256) (2535,)


In [10]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def build_mlp(hp, input_dim, n_classes):
    model = keras.Sequential()

    model.add(layers.Input(shape=(input_dim,)))

    n_layers = hp.Int("n_layers", 1, 5)

    for i in range(n_layers):
        units_exp = hp.Int(f"units_exp_{i}", 6, 12, step=1)
        model.add(layers.Dense(2**units_exp, activation="relu"))

        dropout = hp.Float(f"dropout_{i}", 0.05, 0.45, step=0.1)
        model.add(layers.Dropout(dropout))

    model.add(layers.Dense(n_classes, activation="softmax"))

    lr = hp.Choice("lr", [1e-2, 1e-3, 5e-4, 1e-4])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [10]:
import keras_tuner as kt

def run_tuner(X, y, input_dim, n_classes, project_name):

    def model_builder(hp):
        return build_mlp(hp, input_dim, n_classes)

    tuner = kt.BayesianOptimization(
        hypermodel=model_builder,
        objective="val_loss",
        max_trials=20,
        directory="tuning_results",
        project_name=project_name,
        overwrite=True
    )

    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    tuner.search(
        X, y,
        validation_split=0.2,
        epochs=30,
        batch_size=64,
        callbacks=[early_stop],
        verbose=1
    )

    best_model = tuner.get_best_models(1)[0]
    best_hp = tuner.get_best_hyperparameters(1)[0]

    return best_model, best_hp

In [11]:
n_classes = len(np.unique(y))

results = {}

datasets = {
    "LBP": (X_train_LBP, X_train_LBP.shape[1]),
    "IMG16": (X_train_16, X_train_16.shape[1]),
    "IMG32": (X_train_32, X_train_32.shape[1]),
}

for name, (X, dim) in datasets.items():
    print(f"\n=== Tuning: {name} ===")

    model, hp = run_tuner(
        X, y_train,
        input_dim=dim,
        n_classes=n_classes,
        project_name=name
    )

    results[name] = {
        "model": model,
        "hp": hp.values
    }


Trial 20 Complete [00h 00m 20s]
val_loss: 1.162202000617981

Best val_loss So Far: 1.1295034885406494
Total elapsed time: 00h 30m 04s


In [12]:
from sklearn.metrics import accuracy_score
import numpy as np

testset_X = {
    "LBP": X_test_LBP,
    "IMG16": X_test_16,
    "IMG32": X_test_32
}
dataset_X = {
    "LBP": X_train_LBP,
    "IMG16": X_train_16,
    "IMG32": X_train_32
}



for name, (X, dim) in datasets.items():
    model = results[name]["model"]

    print(f"\n{name}")
    print(f"Best HP: {results[name]['hp']}")

    y_train_pred = model.predict(dataset_X[name], verbose=0)
    y_test_pred  = model.predict(testset_X[name], verbose=0)
    y_train_pred = np.argmax(y_train_pred, axis=1)
    y_test_pred  = np.argmax(y_test_pred, axis=1)

    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc  = accuracy_score(y_test, y_test_pred)

    print(f"train_acc: {train_acc:.4f}")
    print(f"test_acc : {test_acc:.4f}")


LBP
Best HP: {'n_layers': 3, 'units_exp_0': 12, 'dropout_0': 0.45, 'lr': 0.0005, 'units_exp_1': 12, 'dropout_1': 0.15000000000000002, 'units_exp_2': 12, 'dropout_2': 0.05, 'units_exp_3': 12, 'dropout_3': 0.35000000000000003, 'units_exp_4': 8, 'dropout_4': 0.25}
train_acc: 0.6993
test_acc : 0.6927

IMG16
Best HP: {'n_layers': 4, 'units_exp_0': 12, 'dropout_0': 0.15000000000000002, 'lr': 0.0001, 'units_exp_1': 11, 'dropout_1': 0.25, 'units_exp_2': 8, 'dropout_2': 0.15000000000000002, 'units_exp_3': 7, 'dropout_3': 0.15000000000000002, 'units_exp_4': 7, 'dropout_4': 0.35000000000000003}
train_acc: 0.7438
test_acc : 0.7124

IMG32
Best HP: {'n_layers': 4, 'units_exp_0': 9, 'dropout_0': 0.15000000000000002, 'lr': 0.0005, 'units_exp_1': 7, 'dropout_1': 0.25, 'units_exp_2': 6, 'dropout_2': 0.05, 'units_exp_3': 9, 'dropout_3': 0.05, 'units_exp_4': 7, 'dropout_4': 0.35000000000000003}
train_acc: 0.7390
test_acc : 0.7116


In [16]:
import keras_tuner as kt

best_hp = kt.HyperParameters()

best_hp.Fixed("n_layers", 4)

best_hp.Fixed("units_exp_0", 12)
best_hp.Fixed("dropout_0", 0.15)
best_hp.Fixed("units_exp_1", 11)
best_hp.Fixed("dropout_1", 0.25)
best_hp.Fixed("units_exp_2", 8)
best_hp.Fixed("dropout_2", 0.15)
best_hp.Fixed("units_exp_3", 7)
best_hp.Fixed("dropout_3", 0.15)
best_hp.Fixed("units_exp_4", 7)
best_hp.Fixed("dropout_4", 0.35)
best_hp.Fixed("lr", 1e-4)

model = build_mlp(best_hp, 256, 13)

E0000 00:00:1781710084.067659    7496 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [19]:
early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

history = model.fit(X_train_16,
                     y_train,
                     validation_split=0.2,
                     epochs=30,
                     batch_size=64,
                     callbacks=[early_stop],
                     verbose=1)

Epoch 1/30


127/127 ━━━━━━━━━━━━━━━━━━━━ 15s 120ms/step - accuracy: 0.6892 - loss: 1.3746 - val_accuracy: 0.6824 - val_loss: 1.3518
Epoch 2/30
127/127 ━━━━━━━━━━━━━━━━━━━━ 16s 124ms/step - accuracy: 0.6912 - loss: 1.3166 - val_accuracy: 0.6874 - val_loss: 1.2719
Epoch 3/30
127/127 ━━━━━━━━━━━━━━━━━━━━ 16s 127ms/step - accuracy: 0.6948 - loss: 1.2711 - val_accuracy: 0.6908 - val_loss: 1.2751
Epoch 4/30
127/127 ━━━━━━━━━━━━━━━━━━━━ 16s 122ms/step - accuracy: 0.6977 - loss: 1.2328 - val_accuracy: 0.6893 - val_loss: 1.2253
Epoch 5/30
127/127 ━━━━━━━━━━━━━━━━━━━━ 17s 130ms/step - accuracy: 0.7019 - loss: 1.2077 - val_accuracy: 0.6903 - val_loss: 1.2075
Epoch 6/30
127/127 ━━━━━━━━━━━━━━━━━━━━ 18s 140ms/step - accuracy: 0.7045 - loss: 1.1813 - val_accuracy: 0.6889 - val_loss: 1.2163
Epoch 7/30
127/127 ━━━━━━━━━━━━━━━━━━━━ 18s 141ms/step - accuracy: 0.7046 - loss: 1.1758 - val_accuracy: 0.6938 - val_loss: 1.2928
Epoch 8/30
127/127 ━━━━━━━━━━━━━━━━━━━━ 17s 132ms/step - accuracy: 0.7076 - loss: 1.1560 - val

In [20]:
model.save("mlp.keras")